<p align="center">
  <img src="https://img.shields.io/badge/Research%20Mode-ON-4cbb17?style=for-the-badge" alt="Research Mode">
</p>


# Substantia Nigra Cell Subtyping 
## Case Study — Part 01: Set Up and Data Preparation

**ASAP CRN Learning Lab**  
Reproducible exploratory and meta-analysis examples using ASAP CRN data

---

### Overview

This notebook is the first part of a multi-part case study demonstrating a reproducible workflow for **cell subtyping in the human Substantia Nigra** using a **ASAP CRN single-cell cohort dataset**. The Substantia Nigra is a primary site of **dopaminergic neuron degeneration in Parkinson’s disease**, making it a biologically relevant region for cell subtyping analyses.

In this setup step, we establish the analysis environment, define dataset context, and construct a **raw AnnData object** that will serve as the foundation for all downstream preprocessing, integration, and annotation steps.

---

### Learning Objectives

By the end of this notebook, you will be able to:
- Understand the biological and analytical context of the Substantia Nigra case study
- Configure the workspace and analysis environment
- Assemble input data into a standardized AnnData object
- Persist a data artifact for reproducible downstream analysis

---


### Prerequisites

- Approved access to the relevant **ASAP CRN data collections** via the **ASAP CRN Cloud**
- A working analysis environment (e.g., Verily Workbench) with required dependencies installed

---

### Inputs

- An ASAP CRN single-cell harmonized cohort dataset accessed through the ASAP-CRN Cloud  
- Metadata tables required for AnnData construction (e.g., sample- and cell-level annotations)

---

### Outputs

This notebook generates the following AnnData artifacts:

- **Curated Substantia Nigra AnnData object (full gene space)**  
  - Example: `asap-{dataset_team}__sn_cells__full_genes__curated.h5ad`  
  - Contains Substantia Nigra–derived cells with full gene expression and integrated metadata  
  - Serves as the primary, reusable input for downstream case study notebooks  

- **Intermediate Substantia Nigra AnnData object (HVG-restricted)**  
  - Example: `asap-{dataset_team}__sn_cells__hvg.h5ad`  
  - Contains Substantia Nigra–derived cells restricted to highly variable genes (HVGs)  
  - Includes normalized and log-transformed expression values, along with PCA and other preprocessing results  

> The HVG-restricted object is saved as an intermediate artifact to support efficient integration and clustering workflows, while the full-gene object is retained for downstream biological interpretation and analysis.

---

### Notes

- This case study is organized as a **sequential, multi-part workflow**. Each notebook builds on outputs generated in previous parts and is intended to be run in order.
- Run all cells sequentially for the smoothest setup experience. You can return later to explore, adapt, or extend individual steps.

> This notebook is part of the ASAP CRN Learning Lab and is designed to be executed using approved data accessed through the ASAP CRN Cloud.



## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Package Imports and Configuration](#2-package-imports-and-configuration)
3. [Data Sources and Context](#3-data-sources-and-context)
4. [Data Assessment and Filtering](#4-data-assessment-and-filtering)
5. [Data Export](#5-data-export)


## 1. Environment Setup

This case study is designed to run in a **Conda environment** defined by the provided `environment.yml`.
Using this environment ensures consistent package versions and reproducible results across the ASAP-CRN Learning Lab.

### Step 1 — Open a terminal in Verily Workbench
From your Verily Workbench workspace app:
- Click the **+** to open a new window or navigate to Launcher window
- Launch **Terminal**

This opens a shell session within your workspace where you can run the commands below.

### Step 2 — Create and activate the Conda environment
```bash 
    conda env create -f environment.yml 
    conda init
    #close and reopen a terminal
    conda activate sn_celltyping
```

### Step 3 —  Register the environment as a Jupyter kernel
```bash
python -m ipykernel install --user \
  --name sn_celltyping \
  --display-name "Python (sn_celltyping)"
```

After completing these steps, return to JupyterLab and select **Python (pilot_workshop)** as your notebook kernel.

> Notes
> - These steps only need to be run **once** per environment.
> - In managed analysis environments (e.g., Verily Workbench), the Conda environment and kernel may already be available or configured differently.
> - If you encounter package issues, ensure the environment is activated before launching Jupyter.

## 2. Package Imports and Configuration

In [ ]:
# Core scientific computing and visualization libraries
import numpy as np
import pandas as pd
import scanpy as sc

# Standard library imports
import sys
import subprocess
import importlib
import warnings
import os
from pathlib import Path

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime


## 3. Data sources and Context

### 3.1 Set dataset paths
In this example, we are working with the **PMDBS single‑cell RNA‑seq cohort** dataset:

- **Workflow** → `pmdbs_sc_rnaseq`  
- **Team** → `cohort`  
- **Source** → `pmdbs`  
- **Type** → `sc-rnaseq`  

These components are combined to construct the bucket and dataset names.  
We then set the path to the **cohort analysis outputs** and preview the available files.


In [ ]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

In [ ]:
## Build and set path to desired dataset
DATASETS_PATH = WS_ROOT / "01_PMDBS" / "pmdbs-sc-rnaseq-v3"

workflow       = "pmdbs_sc_rnaseq"
dataset_team   = "cohort"
dataset_source = "pmdbs"
dataset_type   = "sc-rnaseq"

bucket_name  = f"{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH / bucket_name / workflow
print("Dataset Path:", dataset_path)

cohort_analysis_path = dataset_path / "cohort_analysis"
print("Contents of cohort_analysis:")
!ls {cohort_analysis_path}


In [ ]:
# Define a local path for workshop files
local_data_path = WS_FILES / "sn_celltyping"

# map my cells directories
mapmycells_input_dir = ( local_data_path / "mapmycells/input" )
mapmycells_output_dir = ( local_data_path / "mapmycells/output" )

# other directories
resources_path =( local_data_path / "resources")
plots_path =( local_data_path / "output_plots")
output_path =( local_data_path / "output_tables")

# Make sure the directories exists
os.makedirs(mapmycells_input_dir, exist_ok=True)
os.makedirs(mapmycells_output_dir, exist_ok=True)
os.makedirs(resources_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)
os.makedirs(plots_path, exist_ok=True)

# Create the directory if it doesn't already exist
if not local_data_path.exists():
    local_data_path.mkdir(parents=True)

print(f"Local data directory ready at: {local_data_path}")

### 3.2.  Copy Data Locally
In this step, we retrieve the curated input files required for the case study:

- **`asap-cohort.final_metadata.csv`** → cell‑level metadata table
- **`asap-cohort.final.h5ad`** → AnnData object containing HVG expression data and associated annotations

These files are copied into the local `pilot_workshop_files` directory (if not already present) and prepared for analysis.

The metadata file is loaded into a Pandas DataFrame, while the .h5ad file is opened as an AnnData object in backed mode to enable efficient access to large datasets without loading the entire object into memory.

In [ ]:
# Downloading obs field (cell metadata)
# Define the expected local path for the metadata file.
cell_metadata_local_path = local_data_path / f"asap-{dataset_team}.final_metadata.csv"\

# Check if the metadata file already exists locally.
if not cell_metadata_local_path.exists():
    # Construct the original path where the metadata file is stored.
    cell_metadata_og_path = cohort_analysis_path / f"asap-{dataset_team}.final_metadata.csv"

    # Use a shell command (`cp`) to copy the file from the original location
    # into the local workshop_files directory for analysis.
    !cp {cell_metadata_og_path} {cell_metadata_local_path}

In [ ]:
# Downloading the anndata object
# Define the expected local path
adata_local_path = local_data_path / f"asap-{dataset_team}.final.h5ad"

# Check if the adata file already exists locally.
if not adata_local_path.exists():
    # Construct the original path where the metadata file is stored.
    adata_cell_metadata_og_path = cohort_analysis_path / f"asap-{dataset_team}.final.h5ad"

    # Use a shell command (`cp`) to copy the file from the original location
    # into the local workshop_files directory for analysis.
    !cp {adata_cell_metadata_og_path} {adata_local_path}

# load the adata object
adata = sc.read_h5ad(adata_local_path, backed="r")
adata

> ⏱️ **Expected runtime:** ~8–10 minutes depending on dataset size and available compute.

### 3.3 Metadata Access and Integration

Here we load and integrate cell-level metadata with the AnnData object. This step ensures that biological and technical annotations are correctly aligned with expression data and available for downstream analysis and interpretation.



In [ ]:
#Define metadata folder path
ds_metadata_path = WS_ROOT / "release_resources/cohort-pmdbs-sc-rnaseq/metadata"

#preview contents
!ls {ds_metadata_path} 

In [ ]:
# study-level metadata
STUDY = pd.read_csv(ds_metadata_path / "STUDY.csv", index_col = 0)
STUDY[["ASAP_team_id" , "dataset_name","sample_types"]]

In [ ]:
# Sample-level metadata
SAMPLE = pd.read_csv(ds_metadata_path / "SAMPLE.csv", index_col=0)
# Subject-level metadata
SUBJECT = pd.read_csv(ds_metadata_path / "SUBJECT.csv", index_col=0)
#  Brain-sample metadata
PMDBS = pd.read_csv(ds_metadata_path / "PMDBS.csv", index_col=0)
# Experimental condition metadata
CONDITION = pd.read_csv(ds_metadata_path / "CONDITION.csv", index_col=0)

# Select Relevant Columns
sample_cols = [
    "ASAP_sample_id",
    "ASAP_subject_id",
    "ASAP_team_id",
    "ASAP_dataset_id",
     "subject_id",
    "replicate",
    "condition_id",
    "age_at_collection"
]
subject_cols = [
    "ASAP_dataset_id",
    "ASAP_subject_id",
    "subject_id",
    "source_subject_id",
    "sex",
    "biobank_name",
    "primary_diagnosis",
]
pmdbs_cols = [
    "ASAP_sample_id",
    "brain_region",
    "region_level_1",
    "region_level_2",
    "region_level_3",
]
condition_cols = [
    "ASAP_team_id",
    "ASAP_dataset_id",
    "condition_id",
    "intervention_name",
    "intervention_id",
    "protocol_id",
]

In [ ]:
#some scherzer samples have NA for condition but filled intervention_id 
CONDITION["condition_id"] = CONDITION["condition_id"].fillna(CONDITION["intervention_id"])

In [ ]:
df = pd.merge(
    SAMPLE[sample_cols],
    CONDITION[condition_cols],
    on=["ASAP_dataset_id","ASAP_team_id", "condition_id"],
    how="left",          # keep all SAMPLE rows, add CONDITION info
    validate="many_to_one"  # each SAMPLE row maps to one CONDITION row
)

df = pd.merge(
    df,
    SUBJECT[subject_cols],
    on=["ASAP_subject_id", "ASAP_dataset_id", "subject_id"],
    how="left",          # keep all SAMPLE rows, add SUBJECT info
    validate="many_to_many"  # each SUBJECT row maps to multiple SAMPLE rows
)

# Merge in brain-region information
df = pd.merge(df, 
              PMDBS[pmdbs_cols], 
              on=["ASAP_sample_id"], 
              how="left", 
              validate = "many_to_many" )

# create unique sample identifier
df["sample"] = df["ASAP_sample_id"] + "_" + df["replicate"]


In [ ]:
# Recode brain region to be "PFC", "MFG", "HIP", "SN", "ACG", "IPL, "AMG", "PUT"
brain_fix = {
    "Prefrontal Cortex": "PFC",
    "Middle_Frontal_Gyrus": "MFG",
    "Hippocampus": "HIP",
    "Substantia_Nigra ": "SN",
    "ACG": "ACG",
    "IPL": "IPL",
    "Middle temporal gyrus": "MTG",
    "Substantia nigra": "SN",
    "Prefrontal cortex": "PFC",
    "Amygdala": "AMG",
    "Putamen": "PUT",
}
df["brain_region"] = df["brain_region"].map(brain_fix)

In [ ]:
 df["region_level_2"].value_counts()

In [ ]:
# Map to find more course designations
brain_simple = {
    "PFC": "frontal_ctx",
    "MFG": "frontal_ctx",
    "ACG": "cingulate_ctx",
    "IPL": "parietal_ctx",
    "MTG": "temporal_ctx",
    "HIP": "subcortical",
    "AMG": "subcortical",
    "PUT": "subcortical",
    "SN": "subcortical",
}

df["brain_region_simple"] = df["brain_region"].map(brain_simple)


# Define sample to match
br_mapper_full = dict(zip(df["sample"], df["brain_region"]))
br_mapper_simple = dict(zip(df["sample"], df["brain_region"].map(brain_simple)))

# Parkinsons and control samples
condition_id_mapper = dict(zip(df["sample"], df["condition_id"]))
case_id_mapper = dict(zip(df["sample"], df["intervention_name"]))

# Detailed brain region mapper 
region_1_mapper = dict(zip(df["sample"], df["region_level_1"]))
region_2_mapper = dict(zip(df["sample"], df["region_level_2"]))

# Diagnoses
diagnoses_mapper = dict(zip(df["sample"], df["primary_diagnosis"]))

#dataset metadata
dataset_mapper = dict(zip(df["sample"], df["ASAP_dataset_id"]))
biobank_mapper = dict(zip(df["sample"], df["biobank_name"]))

In [ ]:
dataset_metadata_file = local_data_path / "asap-cohort-dataset-metadata.csv"
df.to_csv(dataset_metadata_file)

In [ ]:
# Map samples to metadata
adata.obs["brain_region"] = adata.obs["sample"].map(br_mapper_full)
adata.obs["brain_region_simple"] = adata.obs["sample"].map(br_mapper_simple)
adata.obs["case_id"] = adata.obs["sample"].map(case_id_mapper)
adata.obs["condition_id"] = adata.obs["sample"].map(condition_id_mapper)
adata.obs["region_level_1"] = adata.obs["sample"].map(region_1_mapper)
adata.obs["region_level_2"] = adata.obs["sample"].map(region_2_mapper)
adata.obs["dataset_id"] = adata.obs["sample"].map(dataset_mapper)
adata.obs["biobank_name"] = adata.obs["sample"].map(biobank_mapper)

## 4. Data Assessment and Filtering 

At this stage, all relevant metadata have been integrated and mapped to standardized fields. We now assess the composition of the dataset to understand **anatomical coverage, dataset provenance, and condition balance**, with a focus on relevance to the **Substantia Nigra (SN)**.

This assessment informs which samples and datasets are appropriate to carry forward for downstream subtyping analyses.


### 4.1 Anatomical Coverage and Substantia Nigra Specificity

We first evaluate the anatomical distribution of samples using standardized brain region mappings.  

Both detailed and simplified brain region annotations are used to assess whether datasets include sufficient **Substantia Nigra** representation for this case study.


In [ ]:
df[["ASAP_team_id","region_level_2"]].value_counts()

In [ ]:
print(df[df["region_level_2"] == "Substantia nigra"][["ASAP_team_id", "ASAP_dataset_id"]].value_counts())

#validating sample ID is unique
df.duplicated(subset="ASAP_sample_id").sum()
print(len(PMDBS[PMDBS["region_level_2"]== "Substantia nigra"]), "SN samples out of" ,len(PMDBS), "total samples")

In [ ]:
df[df["region_level_2"] == "Substantia nigra"][["ASAP_team_id", "biobank_name"]].value_counts()

In [ ]:
# Check missingness and most common values
print("n_obs:", adata.n_obs)
print("missing region_level_2:", adata.obs["region_level_2"].isna().sum())

adata.obs["region_level_2"].value_counts(dropna=False).head(25)


In [ ]:
# identify substantia nigra cells
sn_cells = adata.obs["region_level_2"] == "Substantia nigra"
# Final boolean mask for subsetting
include = sn_cells
print(sn_cells.sum())

### 4.2 Subsetting to Substantia Nigra Cells
Based on the dataset assessment, we restrict the analysis to cells derived from the **Substantia Nigra (SN)**. The SN is a primary site of **dopaminergic (DA) neuron degeneration in Parkinson’s disease**, making it a biologically relevant region for cell subtyping analyses focused on disease mechanisms.

By subsetting to SN-derived cells, we enrich for neuronal populations of interest, particularly dopaminergic neurons, while reducing confounding signal from unrelated brain regions.


In [ ]:
# create SN subset
sn_ad = adata[include].to_memory()
adata.file.close()  # close the original adata file

In [ ]:
sn_ad.obs[["dataset_id", "biobank_name", "condition_id", "region_level_2"]].value_counts()

In [ ]:
sn_ad.obs[["class_name"]].value_counts()

In [ ]:
# We will save this intermediate anndata which will contain:
# - Cells restricted to the Substantia Nigra (SN)
# - Highly variable genes (HVGs) only
# - Normalized and log-transformed expression values
# - PCA and other preprocessing results stored in .obsm

sn_samples_filename = (
    local_data_path / f"asap-{dataset_team}.sn_hvg.h5ad"
)
sn_ad.write_h5ad(sn_samples_filename)

### 4.3 Integrating with Full Gene Expression Matrix
Up to this point, we have been using a **highly variable gene (HVG)–restricted** AnnData object. To enable downstream analyses that require access to the **full gene expression space** (e.g., marker detection or pathway analysis), we now reconnect the Substantia Nigra–subsetted cells to the complete gene expression matrix.

In this step, we load the full, unfiltered AnnData object and extract expression values for the previously selected Substantia Nigra cells. These values are combined with curated metadata and embeddings from the HVG-based object to construct a new AnnData object that preserves full gene coverage while maintaining consistent cell annotations.

In [ ]:
# Define paths to the full, unfiltered AnnData object
full_adata_filename = (
    cohort_analysis_path / f"asap-{dataset_team}.merged_cleaned_unfiltered.h5ad"
)
l_full_adata_filename = (
    local_data_path / f"asap-{dataset_team}.merged_cleaned_unfiltered.h5ad"
)
# Copy full AnnData locally if not already present

if not l_full_adata_filename.exists():
    !cp {full_adata_filename} {l_full_adata_filename}

> ⏱️ **Expected runtime:** ~3-5 minutes depending on dataset size and available compute when run for the first time.

In [ ]:
# Load the full gene expression matrix in backed mode
# (avoids loading the entire object into memory)var_ = full_adata.var.copy()

full_adata = sc.read_h5ad(l_full_adata_filename, backed="r")

> ⏱️ **Expected runtime:** ~5-7 minutes depending on dataset size and compute resources.

In [ ]:
# Preserve full gene annotations
var_ = full_adata.var.copy()

# Extract expression values for Substantia Nigra cells only
# using cell identifiers from the HVG-based AnnData object
X = full_adata[sn_ad.obs_names].X.copy()

# Close the backing file explicitly
full_adata.file.close()

> ⏱️ **Expected runtime:** ~5-6 minutes depending on dataset size and compute resources.

In [ ]:
# Construct a new AnnData object with:
# - full gene expression values
# - SN-restricted cells
# - curated metadata and embeddings from the HVG-based object

sn_full_ad = sc.AnnData(
    X=X,
    obs=sn_ad.obs,
    var=var_,
    uns=sn_ad.uns,
    obsm=sn_ad.obsm,
)
sn_full_ad

## 5. Data Export

In this step, we export the **Substantia Nigra subsetted AnnData object** constructed in this notebook. This artifact reflects all metadata integration, anatomical filtering, and dataset selection decisions made in earlier sections.

The exported AnnData object serves as a reproducible input for subsequent parts of the case study and enables downstream preprocessing, integration, and annotation workflows to be run consistently.

This AnnData object contains:
- Cells restricted to the **Substantia Nigra (SN)**
- The **full gene expression matrix** for SN-derived cells
- Curated metadata and annotations stored in `.obs`
- Embeddings and preprocessing results (e.g., PCA, UMAP) carried forward from HVG-based analyses and stored in `.obsm`

> Notes
> - An HVG-restricted AnnData object is exported earlier for integration and clustering workflows.
> - The object exported here restores full gene coverage for downstream biological interpretation.



In [ ]:
# To avoid overwriting or losing previously computed results,
# we archive selected embeddings by renaming them with a leading underscore.
# This allows us to retain historical results while freeing the original keys
# for recomputation or alternative methods downstream.


for key in ['X_pca',
            'X_pca_harmony',
            'X_scANVI',
            'X_scVI',
            'X_umap']:
    archived_key = f"_{key}"
    # rename
    sn_full_ad.obsm[archived_key] = sn_full_ad.obsm.pop(key)
    

# Similarly, we archive selected cell-level annotations in `.obs`
# to preserve earlier labels, probabilities, and clustering results.
# Archived fields are prefixed with an underscore to indicate
# they reflect a previous analysis state.

old_obs_fields = [ 'cell_type',
 'phenotype',
 'rho',
 'prob',
 'class_name',
 'subclass_name',
 'supertype_name',
 '_scvi_batch',
 '_scvi_labels',
 'C_scANVI',
 'leiden_res_0.05',
 'leiden_res_0.10',
 'leiden_res_0.20',
 'leiden_res_0.40'
 ]

for key in old_obs_fields:
    archived_key = f"_{key}"
    # rename
    sn_full_ad.obs[archived_key] = sn_full_ad.obs.pop(key)

In [ ]:
# Save the curated AnnData object
sn_full_samples_filename = (
    local_data_path / f"asap-{dataset_team}.sn_cells__full_genes_curated.h5ad"
)
sn_full_ad.write_h5ad(sn_full_samples_filename)

> ⏱️ **Expected runtime:** ~4-5 minutes depending on dataset size and compute resources.

---

## Summary and Next Steps

In this notebook, we established the foundational inputs for Substantia Nigra cell subtyping by integrating metadata, assessing dataset composition, and constructing curated AnnData objects restricted to Substantia Nigra–derived cells.

The exported AnnData artifact provides a stable, reproducible handoff for downstream analyses, including preprocessing, integration, and cell-type annotation workflows.

### Continue the Case Study

- Proceed to **Part 02: Processing and Feature Selection** to normalize expression data and identify highly variable genes.
- Refer to the **ASAP-CRN Learning Lab documentation** for additional context, workflows, and best practices.

> This notebook is part of the ASAP-CRN Learning Lab and is intended to be executed using approved data accessed through the ASAP-CRN Cloud.

